# Disney Cruise Line (India) - scrape, clean, answer

**Please read - how this notebook works (honest note).**
`disneycruise.disney.go.com` blocks datacenter IPs (Google Colab included) with an *Access Denied* page,
so the live scraper **cannot run inside Colab**. I ran it on my own machine with Playwright + Microsoft Edge
(`src/disney_cruise_scraper.py`): accept consent -> click **View Dates** -> scroll until all **171 cards / 35 pages** loaded.

The raw output of that run (`raw.json`) is embedded below (gzip + base64). Everything after that - the cleaning rules,
`results.csv` and the five answers - is the **same code** as in the `.py` file and runs here for real.

Run all cells: *Runtime -> Run all*.

## 1. Scraper code (reference only - runs locally, not in Colab)

```python
def scrape():
    """Open the site, accept consent, click View Dates, scroll, collect cards."""
    from playwright.sync_api import TimeoutError as PWTimeout, sync_playwright

    api_payloads = []

    with sync_playwright() as p:
        # Edge + visible window: the only combination that passes bot protection.
        browser = p.chromium.launch(channel="msedge", headless=False)
        page = browser.new_context(viewport={"width": 1400, "height": 900}).new_page()

        # Keep the site's own JSON feed as a cross-check (totalPages, totals).
        def on_response(resp):
            if "available-products" in resp.url:
                try:
                    api_payloads.append({"url": resp.url, "body": resp.json()})
                except Exception:
                    pass
        page.on("response", on_response)

        # Step 1: open the home page and accept the cookie/consent banner.
        page.goto(BASE_URL, wait_until="domcontentloaded", timeout=90_000)
        try:
            page.locator("#onetrust-accept-btn-handler").click(timeout=20_000)
            print("Consent accepted")
        except PWTimeout:
            print("No consent banner shown")

        # Step 2: "View Dates" with the default filters (any date / any
        # destination / any port / 2 guests) which lists every cruise.
        page.locator(".view-cruises-button").first.click(timeout=30_000)

        # Step 3: wait for the first cards to render.
        page.wait_for_selector("dcl-product-card", timeout=60_000)
        header = page.locator("body").inner_text()
        m = re.search(r"([\d,]+)\s+Cruises", header)
        print("Header says:", m.group(0) if m else "n/a")

        # Step 4: scroll until no new cards appear.
        cards = page.locator("dcl-product-card")
        count, stalled = 0, 0
        while stalled < 6:
            page.mouse.wheel(0, 20_000)
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            page.wait_for_timeout(1500)
            now = cards.count()
            if now > count:
                count, stalled = now, 0
                print(f"  loaded {count} cards (~{count // CARDS_PER_PAGE} pages)", end="\r")
            else:
                stalled += 1
        print(f"\nFinished scrolling: {count} cards")

        raw = page.evaluate(EXTRACT_JS)
        browser.close()

    pages = -(-len(raw) // CARDS_PER_PAGE)
    if pages < MIN_PAGES:
        print(f"WARNING: only {pages} pages loaded, expected >= {MIN_PAGES}")

    DATA_DIR.mkdir(exist_ok=True)
    RAW_JSON.write_text(json.dumps(raw, ensure_ascii=False, indent=1), encoding="utf-8")
    RAW_API_JSON.write_text(json.dumps(api_payloads, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Saved {len(raw)} raw cards -> {RAW_JSON}")
    return raw
```

## 2. Raw data scraped locally (embedded)

In [1]:
import base64, gzip, hashlib, json, re
import pandas as pd

RAW_B64 = (
    "H4sIALvewGoC/+1dW2/bRhZ+L9D/QOShdYBpwdtwyL4lzr1xatjZBNuiWIykiTQxxTF4ieMu9mX/yT7t82Kf9jn/ZH/JnqEs2ZZF"
    "ihzOUJSzQAo0IiNxvjnnO9c5/O3bb6y/wn/Wg5znMXvwk/XA++ENn85y6zAteMasD6mYW6c8mdJzkbIHqLx5RJOEpfLuqw/Ehw+L"
    "vz8vaEqTnLGJdcHzmXXCsjzl45yLJLu6NxZjuvj7T9Zvvy8+O4d7yl//79//4yA7QhEOrm4fF2nKkvGlvPryzcnVpxOas+wvoyLP"
    "RSKvnM7EhRX41hP5+QO45W/o7tL83S/Nc1Hk2u2X5oXNlvaaJtMzesEV17hpHfIT68Hyi5F1RGN6mXEqH8XauEifoCBy2y/SJfWL"
    "XIrmY/qR3l4gTawnnE2F9k1cLP5pkrGETigsnn3mY1G9dCdEnu0pLN1ptr8DXrrrI0IUtDbSvunvWHoJj5umlzmfs95xiJDjK6i4"
    "P1DhP6Q5jXlCrZdZTJMJsg7hrx9EmgAFoLbgeJIdIgX+G6h66AUnQh4h7cFxwqaiM6NzDojcROiZSKXZKCYsndCYGcLpCc8Sdmm9"
    "FuJMFGCB6aVF4WflY81EAc9yLHiS15gUG/64Q5Cb3ZKLH4B3pGBfFubFiICYAKShuFxp2BuaZbRAi8enWa1jQgJPu4kaiBRp5SI/"
    "RKEKVIF2qF7QOBYXjCWWSKx8xqwX8A3WKaNZ78oHlp1gc8q3h5BgD0Uqrn7DcOaYjvkHPrYOBc3yzeBYLJlAfGPxxHpHk7EoPsHK"
    "txmxXxL2w3sglcU3VgOx+kbr4HHKc57N4FHiYj7i9KHULwlSDTgRwir23OvRnPfOwQBKFKpESEFXkbnezGuRae4fNhWZ1Tfe4t9K"
    "OAKMXBWH2G2GxnvQd5aCiNCUj0ZsTVie0/gT3ABf24d9Ok7FNGWZ2M4qAEpInPag4HpQ8BUo5c8DFCf8E2fpUEj3kI5E+fOvizHN"
    "ViC1ttwBQY7rGIw/N7DOsWQdSYegXTQ2FEOsUwy6xVOHoO70QioovawBJwRKVjDhuLFfo8DJpiWrHjjlmCyIEHED7YLWlbqaW7jm"
    "hER8FKh4w463x2LTzgFopYYEI88NtVs9rx84e/eZCEGhQ4xp2p6G/e0kLkS+YxsLVIafWmsrc6GNsOMrqKhr0gkzkzfR43iFENwQ"
    "u19S2+J4mYBLhxsGUIVBODB7qlUdb+GgyeEKXeQqeSF4h0m5nvKVoYdClZ4BbE75duvWN1TEEDlYIVh0tpBWMDiWvyNKqJb9KxGL"
    "4A/RXoHxByxnBhktcpHnOfoZDQ8w/XUo/ijmLF6Xv+ZBaARBqK2/ehV8FVmxCCPf7zkrNqCIsxqXCEVKGqheUG6ZLGxHTBULdZEN"
    "8YxKyqFL4m9P0qIAjoMcVyHY8wKzubwjmn5iMfzGZSbFGlRDf3YPVu8hOzDZXNJHnW6TeW4Y7LvIsZGn0rPmGNQOgwkm/a4MQChL"
    "mwoBiOPvF4Q66MbxkI897cYYd+8C7CVaLQGIIm9YCd3hh6uAm4+8iOj2VrDu/q2+pIgYSTji3gpwHUMiQCBCOIyGRbrNl2+kLluC"
    "EjgGg2q1QLGNjugI+Uocomh46tHFp+2sMK5sgfCMddbsbwYZoPFQ4OKvKbGnwyK7GEXO/S96tcYEh9rjSTy4xhhYaogCG5tzyO5j"
    "NrxtcO5hZPu+9rwv3nnmqkXmQpNrpArltjyXt7tUT/VaCfKjsFd7toueIR1GzLcR/gqgMsRPvjy56BrL5tSbu503day75NUwyUSz"
    "oz/HSgZdv2tZdl8GNAVLc2G9g4ejad7Aefc9EEO8o1hv15m1iqiwHQV64LPa/WYk+3Qy9PeUAmYY/F+Vw+wNhz08pS2kbu1Q06uC"
    "JtrONGnzeZdPhqwrDT+p1+oQgkvtvTbBnuWxsI0iEho7IlYvZcsduyFeLdW2pZB1UNSl8bj1fMh6FouUV5/TBIDhD/Z3ZDyGVQVt"
    "7M3gEGED58PxvSsYt/WmcYQclTRkQ2+6XttvI7zU+Up7Uo94/9Yl/9F6OxNzSbZ/+vH0R+sdT6dAWIvscKZohAIbDL1vKsXVmn27"
    "BY6NTzE3g1IbZ6+tqgFpBw4KXLfXzGM/sWa31JeKHxv4yAuxuYLhPTpf1pbNiWysIzvKjRhttGzrMBBp2rCxyuNdJJpAcMw/01St"
    "FKsDkxDiGxsPappTv32HEZC4F+kGgNQBoKM+ZpCLIwJSoT/YIz3FvGrpviP6B83jL/9ItqtMFKLQDkxJzDDhaZ0NlSCpzE/ZWTdv"
    "zz1Snmx4t8PQlPfYySxrPBXSwTgBRGCfiDsQbu7SNNW9QuiVRwRUMpEmB1aoqlRn3fFQ6Bmj4FZJ70HUDDbnE64uvoUHFjFw9nJ+"
    "3FogXY1yYCQpSQwezN2x3QPMCHJVRlC75pKSZiKMziocIVelNbZZYWUoFYXjGY/5eTYq0imyZEbriIIcsaRlvnBXhQlPHssijmNq"
    "mwY1P7LvRDuAC/5NFOxGB3qJwtvhdkW2xzGVTPtEzHmyMBDsvBjFfFwDpI8iAwMHgy5V8AGXwJuRzlbualPE8ORRIeI7Q+rT7WAY"
    "Wzj0muylayNfJUPe9IUmLJ7yYn7bTMKDwvLOu2WqfmVslBbTKZMDjIssY3H2EC1/r2a9Lgr0+wfL5b4ByZhJlX5apOKcGVj2ichz"
    "SQhz6+DRPFv8L6z7DZO/u8Xxdj2gM2IqvHkU0+xsTVOaz62uXvHzmI7BkQc2YReSAA9Oc37GE2a9HLMxPYfFL355xR9ndApKsv7x"
    "y/ElsE1Keb5QifXrP7N8PONnkneuLlTjSJCtcgrCCQxWlXQkV3ZAPxEiGGs/UEL6TVdpbW58zkQKrPZWXIAoPofvnUhY53RlIJ/R"
    "eC65BFmvYJfAlVHK+3gO8lQODAQ6kN+vAbCVgG/dqsYpSR/iIf19+sREMXUfhd13kKOUciK7aurqvW7g9991staTVUqTC0ZanJee"
    "8117UyOV9f/OZNjTThBl6y8xOB/k3pxd8uQ7jRzPHyQn9jhFry+GJAirxAYN3YE2zZm9ewPbKhlKSdXGsIcq48lM9qcNIo/vh8gL"
    "sLGjAQPrwlaHKQKYImNWm6XzYnK7hPaGXVh/FulZk1M5zRW3abLyZ/jJ7zPr/YymH9Dy+VYhfN3FlmlFLMMjZzel4F4c8qakV1cC"
    "bkV1EtEAG0umDyX9pDHPhDEi5toROvYQ7s4vxPJN1L6xnFG3+tbum7trNXZrLUxpQ8AIud5+tM1oQ7ljsasOzMD1jHXH7JHlMeBu"
    "BzaybQWr7jZ8z8TpOeWJgarPYx6PqECL769Znotce6CKOPx3emnhz+q98RFxvtbi26siYdLsK7tCAUFYSbKj3TUNqsnx3RMDSOeh"
    "e698UWQUDTKyMYFnQzPS0Ci1szYR8m397UMDd4zq6NOETScOCv2vzfustVGtfU4JYWTuNQGmvKLXwNMiLb78i271jEiAiMoLy93/"
    "x4wK8iRf52aHvSYu1rslr4qJWaNiYv0GqJYW9R/GVqHHMEBuaGwvSkXe3hHfJfRJ+WTK8rL89ZimIzoR2bVLJFWBZQuxBL/oejry"
    "ichKj3Ppot+0QK/ELPke/tGjJOfTotLr9OVZOuIYa6Tu6Yi7al2xhZT55Skyzx2alKm2WPwCcHyE3aETEJK0GK2k5z2PYzZfXDgs"
    "Uvrln1TolTgP+Z7d67mI+9GF0Z0qfXl2yA/8QR1KMWOZem168cvR/LaziyKlkS5ZhRpkNTQY/FJ7oH7pzkLExnTp2yBc2Bh85r2b"
    "SkvzM3CxrLB/PAMpEgnlKWtihNr66r7sSPMwGdS0qUF1Rsq0LYoisr8etIqMVTkz7eWL+CgkeN/TM2qH1lSMJcEocPAunZB2LuD9"
    "jZ3LvQh97d64c3M+7UgUEHkdg/s9p6UXHm+uPdxot2reo9G0q0ptWMHy8o1nvw7M4cYpS+DmQxGL+Yg3PzOxWh6y3rLP1eecYXsI"
    "cl3H2PbI+HzL9qyeda0brtk0pebbU4km/P3LvyXrlE9YvyWtykc3eK1czK03UFXuSEgQDofSsWR4iky7U/lYTh3CxNhgJrnXEAR9"
    "V4Y+T5NpXCpWXyHRKQVESrfdOpA/9DgtkuyCj88eoqsnW8L1WCxU+0jCNp4BnHleA5pjq82KaQZaf4UJ1KKPA8sBDr5BYrsfdueZ"
    "vMysR3M6Eal1sFwKzy8fNuPDahPV0AJheXrexdr5LrwfJ6pPeX5GlXs+SnAjYm5GzFXa6LsldQ4jf/RCWln6GWhUfKLW6VjknF6T"
    "aDVWDvKIt288+Y5PxRpDNq7pYjnOwMXusAZmqr89uQMTNh4yCpgFyIuC3Zyp0xeNdz4Tp9u38xzkBPbX1YNSNw6sYZd5c3xB1UN8"
    "L5KUxuosGPk+CiLXlDdyT2oJiolMLN+lE3lkX+czV7aVthxFCTjIXmZ3L07G9TVAsZl6BrbarJth2d6BjU6RR+QgTPC+gizg5j6e"
    "mjh2R2lA2JIIRaFtum9qw4y1xzQds1gk9OZOtAhVmu7Fcz4C/slpugpbeDaS+QJJUMX0Gu4NEc6N50HLTF01lKGj1rjah2fZjwvU"
    "sDO/rUEPkBOZ7Iw8YhMO2AKzrcO6EtIu9gt4lVoTBquMY5GO1xONb+h5zDLr4FjMzxnnD5H1Mqfx9ah88DJyMKLj8YxT6+BEzNn6"
    "La/5J9ByYR3IWb6ADAPh5hld3VYJqyun63sGx3ubxNU8biv7lwj4hiMOQtwAUzmk31h8uVQs0ChYu/40z/OUwWrHZ9bB85hmU3iu"
    "76ynQM9SV2eweJnlKlnwGkKWngsRX/PjMn/O4g9AWeh60ObLlN28QfJvWfo4OIVnZjOWTOUObaHZQPZhGZh+Fda0qOnzyNRnYhiL"
    "RQPkAb/em1M5pl4P3PFQaSCHtrlOMMzD4qreQc+DSQL5FmIDs+jJkEYCrxq+eJZzmmTwYcmhFzffnnPOkpmMJEAEWTKny+rHov5z"
    "8+NKJIPQ5MASgxPbemCAhvEskSVvxzdWU9zoPpUNX9JKn135ACWim5ydLudQZhSCRlT+zpjdUONcpEB46xdOZmIiT/bc/vTo8kwk"
    "4s7HVW5bJcjgpEa2vb89nXU9wJX9nhrToQT5GEWO9mPk0UYxvSOf22J+JdFtGvzLVCnLy1SpzAGsFHyjfB/zlLICRPMRSEQiR9R3"
    "EugGcUj1jkUoiLBBO3fBprKK8uyjSCcm4og68/VLPJEm6vaHjwDD4u69L2gxvX2hEjM52sjcRJUeMAOPf3oXmBcMRDi7jPOGiJ2C"
    "sFEIpdLtiMkZNHg/JlkrWX2FjHYlVsRFjv6KQLgP3nuXF9q3d/IJinzk6X/3xapTa5JyQGIsO4EsufMLNr8FuG53SqQfinWz8Vae"
    "hs6ZdfCOQQh5Jzf1K51QUOHDVMC3XWcBilEqPiX87PpKNYwBIKnd6jt2e7O/Cc4bHkDzZGDjd66pWfLNfuxmZ+GIZRkHR+bglI95"
    "fLm+fas1bWmNIvJdy47+49irbdri3/ZWpa05/1/j6zYhFf0psRA5AfJC7b3Mvsox2K7Jxar3M/solK/H6eYwffvN7/8DufWsyL7R"
    "AAA="
)
raw = json.loads(gzip.decompress(base64.b64decode(RAW_B64)).decode("utf-8"))
print(len(raw), "raw cards loaded  (171 cards = 35 pages x 5 cards)")
raw[:2]

171 raw cards loaded  (171 cards = 35 pages x 5 cards)


[{'title': '3-Night Cruise from Singapore',
  'banner': '',
  'offer': 'Guaranteed with Restrictions',
  'locations': [],
  'price': '₹1,09,956',
  'currency': 'INR',
  'dates_button': 'Show 64 Dates'},
 {'title': '4-Night Cruise from Singapore',
  'banner': '',
  'offer': 'Guaranteed with Restrictions',
  'locations': [],
  'price': '₹1,32,920',
  'currency': 'INR',
  'dates_button': 'Show 38 Dates'}]

## 3. Cleaning rules and answers (identical to the `.py` file)

In [2]:
HOLIDAY_BANNERS = {"Very Merrytime", "Halloween on the High Seas"}

COLUMNS = [
    "Title", "Cruise Name", "Nights", "Banner", "Offer", "Departure Port",
    "Locations", "Price (INR)", "Price Text", "Dates Count", "Note",
]


# --------------------------------------------------------------------------
# Cleaning
# --------------------------------------------------------------------------
# "4-Night Langkawi Cruise from Singapore ending in Hong Kong with 2 stops"
TITLE_RE = re.compile(r"^(?P<nights>\d+)-Night\s+(?P<name>.*?)\s+from\s+(?P<port>.+)$", re.I)
# Extras glued onto the departure port: "ending in X", "ending X",
# "with 2 Stops at Castaway Cay".
NOTE_RE = re.compile(r"\s+(ending(?:\s+in)?\s+.+?|with\s+\d+\s+stops?(?:\s+at\s+.+?)?)\s*$", re.I)


def parse_title(title):
    """Split a card title into (nights, cruise name, departure port, note)."""
    m = TITLE_RE.match(title)
    if not m:
        return "", title, title, "No Note"
    port, notes = m.group("port").strip(), []
    # Peel "ending in X" / "with N stops" off the end (they can be stacked).
    while True:
        n = NOTE_RE.search(port)
        if not n:
            break
        notes.insert(0, n.group(1).strip())
        port = port[:n.start()].strip()
    name = re.sub(r"\s*\bCruise$", "", m.group("name").strip(), flags=re.I) or "Standard"
    return m.group("nights"), name, port, "; ".join(notes) or "No Note"


def clean(raw):
    """Turn raw card dicts into a clean, duplicate-free DataFrame with no empty cells."""
    rows = []
    for r in raw:
        title = r.get("title", "").strip()
        if not title:
            continue
        nights, name, port, note = parse_title(title)
        locs = [x for x in (l.strip() for l in r.get("locations", [])) if x]
        locs = list(dict.fromkeys(locs))                 # de-dupe within a card
        # Cards without a "Sailing to" block: location = departure port.
        locations = "; ".join(locs) if locs else port
        digits = re.sub(r"[^\d]", "", r.get("price", ""))
        dates = re.search(r"(\d+)", r.get("dates_button", ""))
        rows.append({
            "Title": title,
            "Cruise Name": name,
            "Nights": int(nights) if nights else 0,
            "Banner": r.get("banner", "").strip() or "No Banner",
            "Offer": r.get("offer", "").strip() or "No Offer",
            "Departure Port": port,
            "Locations": locations,
            "Price (INR)": int(digits) if digits else 0,
            "Price Text": (f'{r.get("price", "")} {r.get("currency", "")}').strip() or "Price Unavailable",
            "Dates Count": int(dates.group(1)) if dates else 1,
            "Note": note,
        })
    df = pd.DataFrame(rows, columns=COLUMNS).drop_duplicates().reset_index(drop=True)
    assert not df.isna().any().any(), "NaN found after cleaning"
    assert not (df.astype(str).apply(lambda c: c.str.strip()) == "").any().any(), "empty cell found"
    return df


# --------------------------------------------------------------------------
# Answers
# --------------------------------------------------------------------------
def answers(df, api=None):
    """Compute and print the five required answers; return them as a dict."""
    # (i) The site's "Pacific Coast Cruises" destination filter returns the
    # cards titled "Pacific Coast". Like the "948 Cruises" header it counts
    # sailings (dates), so we sum the date counts of those cards.
    pacific = df["Title"].str.contains("Pacific Coast", case=False)
    holiday = df["Banner"].isin(HOLIDAY_BANNERS)
    departs = df["Departure Port"].str.contains("Miami|London", case=False, regex=True)
    out = {
        "i_pacific": int(df.loc[pacific, "Dates Count"].sum()),
        "i_pacific_cards": int(pacific.sum()),
        "ii_total_sailings": int(df["Dates Count"].sum()),
        "ii_distinct_cards": len(df),
        "iii_holiday": int(holiday.sum()),
        "iv_more_than_2_dates": int((df["Dates Count"] > 2).sum()),
        "v_miami_london_departures": int(departs.sum()),
    }
    print("\n=========== ANSWERS ===========")
    print(f"(i)   Cruises with Pacific as destination : {out['i_pacific']}"
          f"   ({out['i_pacific_cards']} Pacific Coast cards; the site's Pacific Coast filter shows {out['i_pacific']} Cruises)")
    print(f"(ii)  Total cruises                       : {out['ii_total_sailings']}"
          f"   ({out['ii_distinct_cards']} distinct cruise cards; site header counts sailings/dates)")
    print(f"(iii) Holiday cruises                     : {out['iii_holiday']}")
    print(f"(iv)  Cruises offering more than 2 dates  : {out['iv_more_than_2_dates']}")
    print(f"(v)   Miami / London departure ports      : {out['v_miami_london_departures']}")
    print("===============================")
    return out


## 4. Clean, save `results.csv`, print the answers

In [3]:
df = clean(raw)
df.to_csv("results.csv", index=False, encoding="utf-8-sig")
print(f"Cleaned: {len(raw)} raw -> {len(df)} rows -> results.csv")
print("Empty cells:", int((df.astype(str).apply(lambda c: c.str.strip()) == "").sum().sum()),
      "| NaN:", int(df.isna().sum().sum()), "| duplicates:", int(df.duplicated().sum()))
ans = answers(df)
df.head(10)

Cleaned: 171 raw -> 171 rows -> results.csv
Empty cells: 0 | NaN: 0 | duplicates: 0

=========== ANSWERS ===========
(i)   Cruises with Pacific as destination : 5   (2 Pacific Coast cards; the site's Pacific Coast filter shows 5 Cruises)
(ii)  Total cruises                       : 948   (171 distinct cruise cards; site header counts sailings/dates)
(iii) Holiday cruises                     : 71
(iv)  Cruises offering more than 2 dates  : 67
(v)   Miami / London departure ports      : 0


,Title,Cruise Name,Nights,Banner,Offer,Departure Port,Locations,Price (INR),Price Text,Dates Count,Note
0,3-Night Cruise from Singapore,Standard,3,No Banner,Guaranteed with Restrictions,Singapore,Singapore,109956,"₹1,09,956 INR",64,No Note
1,4-Night Cruise from Singapore,Standard,4,No Banner,Guaranteed with Restrictions,Singapore,Singapore,132920,"₹1,32,920 INR",38,No Note
2,4-Night Langkawi Cruise from Singapore,Langkawi,4,No Banner,No Offer,Singapore,"Langkawi, Malaysia",147692,"₹1,47,692 INR",27,No Note
3,3-Night Baja Cruise from San Diego,Baja,3,No Banner,Guaranteed with Restrictions,San Diego,"Ensenada, Mexico",118303,"₹1,18,303 INR",21,No Note
4,4-Night Baja Cruise from San Diego,Baja,4,No Banner,Guaranteed with Restrictions,San Diego,"Ensenada, Mexico",124776,"₹1,24,776 INR",9,No Note
5,3-Night Baja Cruise from San Diego,Baja,3,Very Merrytime,Guaranteed with Restrictions,San Diego,"Ensenada, Mexico",129140,"₹1,29,140 INR",4,No Note
6,3-Night Baja Cruise from San Diego,Baja,3,No Banner,Guaranteed with Restrictions,San Diego,"Catalina Island, California; Ensenada, Mexico",131479,"₹1,31,479 INR",3,No Note
7,4-Night Baja Cruise from San Diego,Baja,4,No Banner,Guaranteed with Restrictions,San Diego,"Catalina Island, California; Ensenada, Mexico",139377,"₹1,39,377 INR",18,No Note
8,3-Night Bahamian Cruise from Fort Lauderdale,Bahamian,3,No Banner,Guaranteed with Restrictions,Fort Lauderdale,Disney Lookout Cay at Lighthouse Point,140402,"₹1,40,402 INR",3,No Note
9,4-Night Baja Cruise from San Diego,Baja,4,Very Merrytime,Guaranteed with Restrictions,San Diego,"Ensenada, Mexico",146093,"₹1,46,093 INR",1,No Note


## 5. Check against the file produced locally

In [4]:
EXPECTED_SHA256 = "e18b321e875a42863f6e9645c05aff878f5425c02e63cb6a874f91daba5eb1a2"   # sha256 of the results.csv produced on my machine
actual = hashlib.sha256(open("results.csv", "rb").read()).hexdigest()
print("results.csv identical to the locally generated file:", actual == EXPECTED_SHA256)

results.csv identical to the locally generated file: True


## 6. Download `results.csv` (optional)

In [5]:
try:
    from google.colab import files
    files.download("results.csv")
except ImportError:
    print("Not running in Colab - results.csv is in the working directory.")

Not running in Colab - results.csv is in the working directory.
